# Preprocessing — YawDD (Clip Generation + Labelling)

Generates standardised clips for the YawDD driving dataset and derives clip-level drowsiness labels. Clips are extracted at **10 fps, 112×112, 16 frames, stride 8** and saved as `.npz` arrays, with per-subject manifests for the subject-independent splits (Section 3.5).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
INPUT_DIR  = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/YawDD"
OUTPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
DATASET    = "YawDD"

In [ ]:
!pip -q install opencv-python numpy pandas tqdm

In [ ]:
script = r'''
import argparse
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv", ".mpeg", ".mpg", ".webm"}

def list_videos(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS])

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def resample_indices(num_frames: int, src_fps: float, tgt_fps: float) -> np.ndarray:
    if num_frames <= 0:
        return np.array([], dtype=np.int32)
    if src_fps is None or src_fps <= 0:
        return np.arange(num_frames, dtype=np.int32)

    step = src_fps / tgt_fps
    idxs, t = [], 0.0
    while int(round(t)) < num_frames:
        idxs.append(int(round(t)))
        t += step

    idxs = np.array(idxs, dtype=np.int32)
    idxs = np.clip(idxs, 0, num_frames - 1)
    return np.unique(idxs)

def read_all_frames(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frames.append(frame_bgr)
    cap.release()
    return fps, frames

def preprocess_video_to_clips(video_path: Path, out_dir: Path, target_fps: float,
                              clip_len: int, stride: int, size: int, layout: str, video_id: str):
    fps, frames_bgr = read_all_frames(video_path)
    n = len(frames_bgr)
    if n == 0:
        return []

    idxs = resample_indices(n, fps, target_fps)
    if idxs.size < clip_len:
        return []

    resampled = []
    for i in idxs:
        frame = frames_bgr[int(i)]
        frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resampled.append(frame)
    resampled = np.stack(resampled, axis=0).astype(np.uint8)  # (T,H,W,C)

    T = resampled.shape[0]
    clip_paths = []
    clip_idx = 0
    for start in range(0, T - clip_len + 1, stride):
        clip = resampled[start:start + clip_len]  # (16,112,112,3)

        if layout.upper() == "TCHW":
            clip = np.transpose(clip, (3, 0, 1, 2))  # (C,T,H,W)
        elif layout.upper() == "THWC":
            pass
        else:
            raise ValueError("layout must be 'TCHW' or 'THWC'")

        out_path = out_dir / f"{video_id}__{clip_idx:05d}.npz"
        np.savez_compressed(out_path, clip=clip)
        clip_paths.append(str(out_path))
        clip_idx += 1

    return clip_paths

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_dir", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--dataset", default="UNKNOWN")
    ap.add_argument("--target_fps", type=float, default=10.0)
    ap.add_argument("--clip_len", type=int, default=16)
    ap.add_argument("--stride", type=int, default=8)
    ap.add_argument("--size", type=int, default=112)
    ap.add_argument("--layout", choices=["TCHW", "THWC"], default="TCHW")
    args = ap.parse_args()

    in_root = Path(args.input_dir).expanduser().resolve()
    out_root = Path(args.output_dir).expanduser().resolve()
    out_clips = out_root / "clips_npz"
    safe_mkdir(out_clips)

    videos = list_videos(in_root)
    if not videos:
        raise SystemExit(f"No videos found in: {in_root}")

    rows = []
    for vp in tqdm(videos, desc=f"Preprocessing {args.dataset}"):
        rel = vp.relative_to(in_root)
        video_id = "__".join(rel.with_suffix("").parts)

        try:
            clip_paths = preprocess_video_to_clips(
                vp, out_clips, args.target_fps, args.clip_len, args.stride, args.size, args.layout, video_id
            )
            rows.append({"video": str(vp), "clips_written": len(clip_paths)})
        except Exception as e:
            rows.append({"video": str(vp), "clips_written": 0, "error": str(e)})

    df = pd.DataFrame(rows)
    out_root.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_root / "manifest.csv", index=False)

    print("\\nDone.")
    print("Videos scanned:", len(videos))
    print("Total clips written:", int(df["clips_written"].sum()))
    print("Output folder:", out_root)

if __name__ == "__main__":
    main()
'''
with open("preprocess_clips.py", "w") as f:
    f.write(script)

print("✅ wrote preprocess_clips.py")

In [ ]:
!python preprocess_clips.py \
  --input_dir "$INPUT_DIR" \
  --output_dir "$OUTPUT_DIR" \
  --dataset "$DATASET" \
  --target_fps 10 \
  --clip_len 16 \
  --stride 8 \
  --size 112 \
  --layout TCHW

In [ ]:
import os, glob, pandas as pd, numpy as np

clips_dir = os.path.join(OUTPUT_DIR, "clips_npz")
clips = sorted(glob.glob(clips_dir + "/*.npz"))
print("✅ Num clips:", len(clips))
print("Example:", clips[:3])

x = np.load(clips[0])["clip"]
print("Clip shape:", x.shape, "dtype:", x.dtype)  # expect (3,16,112,112) for TCHW

m = pd.read_csv(os.path.join(OUTPUT_DIR, "manifest.csv"))
print("Videos:", len(m))
print("Errors:", m["error"].notna().sum() if "error" in m.columns else 0)
m[m.get("error").notna()].head(10) if "error" in m.columns else None

In [ ]:
import os, glob, re
import pandas as pd

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
clips_dir = os.path.join(OUT, "clips_npz")
clips = sorted(glob.glob(os.path.join(clips_dir, "*.npz")))
print("Num clips:", len(clips))

def label_from_video_id(video_id: str):
    v = video_id.lower()
    if "yawning" in v:
        return 1
    if "normal" in v or "talking" in v:
        return 0
    return None

# Your naming looks like: Female_mirror__1-FemaleNoGlasses-Normal__00000.npz
# We'll set subject_id = the number before the first '-' in the second chunk (e.g. "1" in "1-FemaleNoGlasses-Normal")
def subject_from_video_id(video_id: str):
    parts = video_id.split("__")
    if len(parts) < 2:
        return None
    second = parts[1]  # "1-FemaleNoGlasses-Normal"
    m = re.match(r"^(\d+)-", second)
    return int(m.group(1)) if m else None

rows = []
unknown_label = 0
unknown_subject = 0

for fp in clips:
    base = os.path.basename(fp)              # ...__00000.npz
    video_id = base.rsplit("__", 1)[0]       # drop clip idx -> "...__1-...-Normal"

    y = label_from_video_id(video_id)
    if y is None:
        unknown_label += 1

    sid = subject_from_video_id(video_id)
    if sid is None:
        unknown_subject += 1

    rows.append({
        "dataset": "YawDD",
        "subject_id": sid,
        "video_id": video_id,
        "label": y,          # 0 normal/talking, 1 yawning
        "clip_path": fp
    })

df = pd.DataFrame(rows)

print("Unknown labels:", unknown_label)
print("Unknown subject_id:", unknown_subject)

df_labeled = df.dropna(subset=["label"]).copy()
df_labeled["label"] = df_labeled["label"].astype(int)

out_csv = os.path.join(OUT, "manifest_yawdd_labeled.csv")
df_labeled.to_csv(out_csv, index=False)

print("✅ Wrote:", out_csv, "rows:", len(df_labeled))
print(df_labeled["label"].value_counts())
df_labeled.head()

In [ ]:
import glob
INPUT_DIR  = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/YawDD"

avis = glob.glob(INPUT_DIR + "/**/*.avi", recursive=True)
print("Total .avi found:", len(avis))
print("Example paths:", avis[:5])

In [ ]:
import os, glob

ROOT = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/YawDD"
avis = glob.glob(ROOT + "/**/*.avi", recursive=True)
print("Total AVIs:", len(avis))
print("Example AVI:", avis[0])

# show the common parent folder of all AVIs (the deepest shared path)
common = os.path.commonpath(avis)
print("Common parent:", common)

# list directories at that level
print("\nTop folders inside common parent:")
!ls -lah "{common}"

In [ ]:
import shutil, os

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
if os.path.exists(OUT):
    shutil.rmtree(OUT)
    print("✅ Deleted old output:", OUT)
else:
    print("No existing output to delete.")

In [ ]:
INPUT_DIR  = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/YawDD"
OUTPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
DATASET    = "YawDD"

In [ ]:
!python preprocess_clips.py \
  --input_dir "$INPUT_DIR" \
  --output_dir "$OUTPUT_DIR" \
  --dataset "$DATASET" \
  --target_fps 10 \
  --clip_len 16 \
  --stride 8 \
  --size 112 \
  --layout TCHW

In [ ]:
import re

with open("preprocess_clips.py", "r") as f:
    txt = f.read()

txt = re.sub(r'VIDEO_EXTS\s*=\s*\{[^}]*\}', 'VIDEO_EXTS = {".avi"}', txt, count=1)

with open("preprocess_clips.py", "w") as f:
    f.write(txt)

print("✅ Script now scans AVI only")

In [ ]:
import os, pandas as pd

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
m = pd.read_csv(os.path.join(OUT, "manifest.csv"))

print("Videos scanned:", len(m))
print("Succeeded:", (m["clips_written"]>0).sum())
print("Failed:", (m["clips_written"]==0).sum())
print("Total clips:", int(m["clips_written"].sum()))

In [ ]:
import os, pandas as pd

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
m = pd.read_csv(os.path.join(OUT, "manifest.csv"))

bad = m[m["clips_written"]==0].copy()
bad_out = os.path.join(OUT, "yawdd_bad_videos.csv")
bad.to_csv(bad_out, index=False)

print("✅ Bad videos:", len(bad))
print("Saved to:", bad_out)
bad.head(10)

In [ ]:
import re

with open("preprocess_clips.py", "r") as f:
    txt = f.read()

txt = re.sub(r'VIDEO_EXTS\s*=\s*\{[^}]*\}', 'VIDEO_EXTS = {".avi"}', txt, count=1)

with open("preprocess_clips.py", "w") as f:
    f.write(txt)

print("✅ preprocess_clips.py now scans only .avi")

In [ ]:
import os, glob, re
import pandas as pd

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
clips_dir = os.path.join(OUT, "clips_npz")
clips = sorted(glob.glob(os.path.join(clips_dir, "*.npz")))
print("Num clips:", len(clips))

def label_from_video_id(video_id: str):
    v = video_id.lower()
    if "yawning" in v:
        return 1
    if "normal" in v or "talking" in v:
        return 0
    return None

# Your clip naming example:
# Female_mirror__1-FemaleNoGlasses-Normal__00000.npz
# subject_id is the leading number before first '-'
def subject_from_video_id(video_id: str):
    parts = video_id.split("__")
    if len(parts) < 2:
        return None
    m = re.match(r"^(\d+)-", parts[1])
    return int(m.group(1)) if m else None

rows = []
unknown_label = 0
unknown_subject = 0

for fp in clips:
    base = os.path.basename(fp)
    video_id = base.rsplit("__", 1)[0]  # drop clip idx (e.g. __00000)

    y = label_from_video_id(video_id)
    if y is None:
        unknown_label += 1

    sid = subject_from_video_id(video_id)
    if sid is None:
        unknown_subject += 1

    rows.append({
        "dataset": "YawDD",
        "subject_id": sid,
        "video_id": video_id,
        "label": y,
        "clip_path": fp
    })

df = pd.DataFrame(rows)

print("Unknown labels:", unknown_label)
print("Unknown subject_id:", unknown_subject)

df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

out_csv = os.path.join(OUT, "manifest_yawdd_labeled.csv")
df.to_csv(out_csv, index=False)

print("✅ Wrote:", out_csv, "rows:", len(df))
print("Label counts:\n", df["label"].value_counts())
df.head(10)

In [ ]:
import pandas as pd, os

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
df = pd.read_csv(os.path.join(OUT, "manifest_yawdd_labeled.csv"))

print("Missing subject_id:", df["subject_id"].isna().sum())

In [ ]:
import pandas as pd, os

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
df = pd.read_csv(os.path.join(OUT, "manifest_yawdd_labeled.csv"))

print("Subjects after preprocessing:", df["subject_id"].nunique())
print("Subjects:", sorted(df["subject_id"].unique()))

In [ ]:
import glob, os, re

INPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/YawDD"
avis = glob.glob(INPUT_DIR + "/**/*.avi", recursive=True)

subjects = set()

for p in avis:
    base = os.path.basename(p)
    m = re.match(r"^(\d+)-", base)
    if m:
        subjects.add(int(m.group(1)))

print("Raw AVI subject count:", len(subjects))
print("Subjects:", sorted(subjects))

In [ ]:
import os, glob, re
import pandas as pd

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
clips_dir = os.path.join(OUT, "clips_npz")
clips = sorted(glob.glob(os.path.join(clips_dir, "*.npz")))

def label_from_video_id(video_id: str):
    v = video_id.lower()
    if "yawning" in v:
        return 1
    if "normal" in v or "talking" in v:
        return 0
    return None

rows = []

for fp in clips:
    base = os.path.basename(fp)
    video_id = base.rsplit("__", 1)[0]

    # split by "__"
    # Example:
    # Female_mirror__9-FemaleNoGlasses-Normal
    parts = video_id.split("__")

    gender_block = parts[0]  # Female_mirror or Male_mirror
    info_block = parts[1]    # 9-FemaleNoGlasses-Normal

    m = re.match(r"^(\d+)-", info_block)
    num = m.group(1) if m else None

    if num is not None:
        if "female" in gender_block.lower():
            subject_id = f"F{int(num):02d}"
        else:
            subject_id = f"M{int(num):02d}"
    else:
        subject_id = None

    rows.append({
        "dataset": "YawDD",
        "subject_id": subject_id,
        "video_id": video_id,
        "label": label_from_video_id(video_id),
        "clip_path": fp
    })

df = pd.DataFrame(rows)
df = df.dropna(subset=["label"])

out_csv = os.path.join(OUT, "manifest_yawdd_labeled.csv")
df.to_csv(out_csv, index=False)

print("✅ Wrote:", out_csv)
print("Unique subjects:", df["subject_id"].nunique())
print(sorted(df["subject_id"].unique()))

In [ ]:
import pandas as pd, os

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/YawDD_112_10fps_16f_stride8"
df = pd.read_csv(os.path.join(OUT, "manifest_yawdd_labeled.csv"))

print("Missing subject_id:", df["subject_id"].isna().sum())
print("Label counts:\n", df["label"].value_counts())
print("Yawning %:", (df["label"].mean()*100).round(2))

In [ ]:
subj_counts = df.groupby("subject_id").size().sort_values()
print("Min clips per subject:", int(subj_counts.min()))
print("Max clips per subject:", int(subj_counts.max()))
subj_counts.head(10), subj_counts.tail(10)

In [ ]:
per_subj_pos = df.groupby("subject_id")["label"].mean().sort_values()
per_subj_pos.head(), per_subj_pos.tail()

In [ ]:
import os, glob
from collections import Counter

UTA="/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"
files=[p for p in glob.glob(UTA+"/**/*", recursive=True) if os.path.isfile(p)]
exts=Counter([os.path.splitext(p)[1].lower() for p in files])
print("Extensions:", exts)

video_exts={".mov",".mp4",".avi",".mkv",".mpeg",".mpg",".webm"}
vids=[p for p in files if os.path.splitext(p)[1].lower() in video_exts]
print("Total video files (any ext):", len(vids))
print("First 20 video paths:")
for v in sorted(vids)[:20]:
    print(v)